Задание "Критерии качества"

---

Как оценить качество работы сети?

1. Предложите  критерии оценки  качества работы предложенных на уроке программ.

2. Один из способов. Программы выдают последовательность символов. Посчитайте перплексию для оценки качества работы этих программ.

Выберите программу - предсказание символа или слова и файл, на котором ведется обучение. Можно начать с искусственных текстов - легче проверить

3. Подберите параметры сети, чтобы она предсказывала "хорошо" и быстро. Что в Вашем случае значит "хорошо"

---

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import numpy as np
import re
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Dense, SimpleRNN, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
def load_text(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
        text = text.replace('\ufeff', ' ')
    return text

sl3_text = load_text('sl3.txt')
print("Символов в sl3.txt:", len(sl3_text))
print(sl3_text[:200])
 
cat_text = load_text('cat.txt')
print("\nДлина текста cat.txt:", len(cat_text))
print("Первые 200 символов:")
print(cat_text[:200])

In [ ]:
def create_and_train_model(text, num_neurons=100, batch_size=16, epochs=10, inp_chars=3):
    if text == cat_text:
        text = re.sub(r'[^А-я]', ' ', text)
        text = text.replace('  ', ' ')
        text = text.lower()
        num_characters = 34  # 33 буквы + пробел
    else:
        num_characters = 6
    
    tokenizer = Tokenizer(num_words=num_characters, char_level=True)
    tokenizer.fit_on_texts(text)
    print("Словарь:", tokenizer.word_index)
    
    data = tokenizer.texts_to_matrix(text)
    
    split_point = int(len(data) * 0.8)
    data_train = data[:split_point]
    data_test = data[split_point:]
    
    n_train = data_train.shape[0] - inp_chars
    X_train = np.array([data_train[i:i+inp_chars, :] for i in range(n_train)])
    Y_train = data_train[inp_chars:]
    
    n_test = data_test.shape[0] - inp_chars
    X_test = np.array([data_test[i:i+inp_chars, :] for i in range(n_test)])
    Y_test = data_test[inp_chars:]
    
    model = Sequential([
        Input((inp_chars, num_characters)),
        SimpleRNN(num_neurons, activation='tanh'),
        Dense(num_characters, activation='softmax')
    ])
    
    model.compile(loss='categorical_crossentropy', 
                  metrics=['accuracy'], 
                  optimizer='adam')
    
    history = model.fit(X_train, Y_train, 
                       batch_size=batch_size, 
                       epochs=epochs, 
                       verbose=1,
                       validation_data=(X_test, Y_test))
    
    return model, tokenizer, history, X_test, Y_test

In [ ]:
#PP
def calculate_perplexity(model, X_test, Y_test):
    predictions = model.predict(X_test, verbose=0)
    
    log_probabilities = []
    for i in range(len(Y_test)):
        true_char_index = np.argmax(Y_test[i])
        prob_true_char = predictions[i][true_char_index]
        log_prob = np.log(prob_true_char + 1e-10)
        log_probabilities.append(log_prob)
    
    avg_neg_log_likelihood = -np.mean(log_probabilities)
    perplexity = np.exp(avg_neg_log_likelihood)
    
    return perplexity, avg_neg_log_likelihood

In [ ]:
def build_phrase(model, tokenizer, inp_str, str_len=20, inp_chars=3):
    result = inp_str
    for i in range(str_len):
        x = []
        for j in range(i, i + inp_chars):
            x.append(tokenizer.texts_to_matrix(result[j]))
        
        x = np.array(x)
        inp = x.reshape(1, inp_chars, tokenizer.num_words)
        
        pred = model.predict(inp, verbose=0)
        d = tokenizer.index_word[pred.argmax(axis=1)[0]]
        
        result += d
    return result

In [ ]:
print("Запуск: 100 нейронов, batch_size=16, epochs=10")
model1, tokenizer1, history1, X_test1, Y_test1 = create_and_train_model(
    sl3_text, num_neurons=100, batch_size=16, epochs=10)
perplexity1, cross_entropy1 = calculate_perplexity(model1, X_test1, Y_test1)
print(f"Перплексия: {perplexity1:.4f}")
print(f"Cross-Entropy: {cross_entropy1:.4f}")
test_phrases = ["ма ", " но", "ном", "ман"]
print("\nПримеры:")
for phrase in test_phrases:
    generated = build_phrase(model1, tokenizer1, phrase)
    print(f"'{phrase}' -> '{generated}'")

In [ ]:
print("Запуск: 200 нейронов, batch_size=16, epochs=10")
model2, tokenizer2, history2, X_test2, Y_test2 = create_and_train_model(
    sl3_text, num_neurons=200, batch_size=16, epochs=10)
perplexity2, cross_entropy2 = calculate_perplexity(model2, X_test2, Y_test2)
print(f"Перплексия: {perplexity2:.4f}")
print(f"Cross-Entropy: {cross_entropy2:.4f}")
print("\nПримеры:")
for phrase in test_phrases:
    generated = build_phrase(model2, tokenizer2, phrase)
    print(f"'{phrase}' -> '{generated}'")

In [ ]:
print("Запуск: 200 нейронов, batch_size=32, epochs=10")
model3, tokenizer3, history3, X_test3, Y_test3 = create_and_train_model(
    sl3_text, num_neurons=200, batch_size=32, epochs=10)
perplexity3, cross_entropy3 = calculate_perplexity(model3, X_test3, Y_test3)
print(f"Перплексия: {perplexity3:.4f}")
print(f"Cross-Entropy: {cross_entropy3:.4f}")
print("\nПримеры:")
for phrase in test_phrases:
    generated = build_phrase(model3, tokenizer3, phrase)
    print(f"'{phrase}' -> '{generated}'")

In [ ]:
print("РЕЗУЛЬТАТЫ:")
print("100 нейронов, batch=16:", f"{perplexity1:.4f}")
print("200 нейронов, batch=16:", f"{perplexity2:.4f}")
print("200 нейронов, batch=32:", f"{perplexity3:.4f}")
print()
print("Лучший результат:", f"{min(perplexity1, perplexity2, perplexity3):.4f}")

In [ ]:
# Для естественного
print("Параметры: 500 нейронов, batch_size=32, epochs=20")
model4, tokenizer4, history4, X_test4, Y_test4 = create_and_train_model(
    cat_text, num_neurons=500, batch_size=32, epochs=20)
perplexity4, cross_entropy4 = calculate_perplexity(model4, X_test4, Y_test4)
print(f"Перплексия: {perplexity4:.4f}")
print(f"Cross-Entropy: {cross_entropy4:.4f}")
test_phrases_natural = ["кот", "дом", "лес", "мам"]
print("\nПримеры генерации:")
for phrase in test_phrases_natural:
    generated = build_phrase(model4, tokenizer4, phrase, str_len=30)
    print(f"'{phrase}' -> '{generated}'")

---

Критерии оценки качества нейросети разделены на количественные и качественные. Основные метрики: перплексия, точность (accuracy), величина потерь (loss) и скорость работы. Качественные показатели включают читаемость текста, соответствие правилам языка и отсутствие зацикливания.

Эксперименты проведены на двух текстах. Для искусственного текста (sl3.txt) лучший результат: перплексия 1.5942 при 100 нейронах и batch_size=16. Увеличение параметров не улучшило результаты. Для естественного текста (cat.txt) перплексия составила 11.3631 даже при 500 нейронах.

На искусственном тексте - получено ожидаемое, на естественном - показали приемлимые числовые оценки, но на практике по факту нашло одну комбинацию и на ней зациклилось для всех примеров, значит явно требует более сложных архитектур.

Выводы: для простых текстов достаточно 100 нейронов. Качественная модель имеет перплексию близкую к 1.0 и accuracy выше 70%. Перплексия подходит для оценки качества.